# Build FAISS Similarity Search (smoke)

A 20-minute access check for the hosted workspace. Run every cell from top to bottom (**Run > Run All Cells**).
It embeds ten short sentences with the local embedding model, builds a FAISS index, runs three queries,
runs the workspace health check, and makes one call to the course LLM.

If any cell fails, copy the error into the chat for the trainer.

In [ ]:
import sys, time
from pathlib import Path

KIT = Path.cwd().resolve().parents[1]   # this notebook lives in kit/labs/smoke/
sys.path.insert(0, str(KIT))
sys.path.insert(0, str(Path.cwd()))

import smoke
start = time.perf_counter()
sentences = smoke.load_sentences()
print(len(sentences), "sentences loaded")

## 1. Load the local embedding model and embed the sentences

In [ ]:
from tools.local_models import load_embedder, embedding_model_name

model = load_embedder()
def embed(texts):
    return model.encode(texts, normalize_embeddings=True)

vectors = embed(sentences)
print(embedding_model_name(), "->", vectors.shape)

## 2. Build a FAISS index and run three queries

In [ ]:
index = smoke.build_index(vectors)
for query in smoke.QUERIES:
    print("Q:", query)
    for score, sentence in smoke.search(index, sentences, embed([query])[0], k=3):
        print(f"   {score:.3f}  {sentence}")
    print()

## 3. Workspace health check

In [ ]:
!{sys.executable} {KIT / 'scripts' / 'preflight.py'} --quick

## 4. One call to the course LLM

In [ ]:
from novaassist.llm_client import get_client

llm = get_client()
reply = llm.complete([{"role": "user", "content": "In one sentence, what is a context window?"}],
                     max_completion_tokens=1024)
print("model:", reply.model, "| latency:", round(reply.latency_ms), "ms | tokens:", reply.usage)
print(reply.text)
print(f"Smoke lab finished in {time.perf_counter() - start:.0f} s")